# Block 02 – Tools and the loop, one step at a time

**You'll see:** a tool card, the loop running step by step, and where prebuilt and hand-built graphs really differ.

In [ ]:
# Setup: run this cell first. It finds the repo, makes it importable and loads .env
import os, sys, pathlib

def find_repo_root():
    starts = []
    if "__vsc_ipynb_file__" in globals():                 # VS Code tells us where this notebook is
        starts.append(pathlib.Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
    starts.append(pathlib.Path.cwd().resolve())
    for start in starts:
        for folder in [start, *start.parents]:
            if (folder / "common" / "config.py").exists():
                return folder
            if (folder / "agents-weekend" / "common" / "config.py").exists():
                return folder / "agents-weekend"
    raise RuntimeError(f"Can't find the agents-weekend repo from {starts}. "
                       "Open the agents-weekend folder in VS Code, or set ROOT by hand.")

ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

from common.config import get_llm, get_cheap_llm, DEFAULT_MODEL, CHEAP_MODEL
from common.tracing import langfuse_config, trace_attributes, flush, tracing_enabled
print("main model:", DEFAULT_MODEL, "| cheap model:", CHEAP_MODEL, "| Langfuse:", tracing_enabled())

In [ ]:
def describe(m):
    """One readable line per message."""
    kind = type(m).__name__.replace("Message", "")
    if getattr(m, "tool_calls", None):
        return f"{kind:<6} asks for: " + ", ".join(f"{tc['name']}({tc['args']})" for tc in m.tool_calls)
    text = (m.text or "").replace("\n", " ")
    return f"{kind:<6} {text[:90]}{'...' if len(text) > 90 else ''}"

## 1. A tool card
Claude never sees your code, only this.

In [ ]:
from common.helpdesk import check_system_status as t
print("NAME:       ", t.name)
print("DESCRIPTION:", t.description)
print("INPUTS:     ", t.args)

## 2. Build the hand-built graph and look at it

In [ ]:
from common.loader import load_block
b02 = load_block("02-tools")
graph = b02.build_graph(get_llm())
print(graph.get_graph().draw_mermaid())

## 3. Run the loop step by step
`stream_mode='updates'` yields what each box added, in order.

In [ ]:
q = "Is the VPN down right now? It keeps dropping."
total = 1
print("[0] Human ", q)
with trace_attributes(tags=["notebook-02"]):
    for chunk in graph.stream({"messages": [{"role": "user", "content": q}]},
                              config=langfuse_config(recursion_limit=12), stream_mode="updates"):
        for node, update in chunk.items():
            print(f"\n--- after box: {node} ---")
            for m in update["messages"]:
                print(f"[{total}]", describe(m)); total += 1

## 4. Prebuilt runs the same steps
Same graph shape → same execution.

In [ ]:
prebuilt = b02.build_prebuilt(get_llm())
steps = [node for chunk in prebuilt.stream({"messages": [{"role": "user", "content": q}]},
                                           config=langfuse_config(recursion_limit=12), stream_mode="updates")
         for node in chunk]
print("prebuilt steps:", steps)

## 5. Where they really differ: a rule enforced in code
The custom graph replaces the `tools` box with `tools_with_policy`.

In [ ]:
pvc = load_block("02-tools", "prebuilt_vs_custom.py")
print(pvc.QUESTION)
pvc.run("prebuilt", pvc.build_prebuilt())
pvc.run("custom", pvc.build_custom())

In [ ]:
flush()